# Diabetes Dataset - Ingestion Pipeline

In [1]:
pip install psycopg2-binary


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import psycopg2
from pathlib import Path

## Connection settings

In [16]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'labdb',
    'user': 'labuser',
    'password': 'labpass'
}

CLEANED_PATH = Path('../raw/uci-diabetes/cleaned/diabetes-cleaned.csv')

## Database connection class

In [17]:
class PostgresDatabase:
    """Class responsible for PostgreSQL connection management."""

    def __init__(self, db_config):
        self.config = db_config
        self.connection = None
        
    def connect(self):
        if self.connection is None:
            try:
                self.connection = psycopg2.connect(
                    host=self.config['host'],
                    port=self.config['port'],
                    dbname=self.config['database'],
                    user=self.config['user'],
                    password=self.config['password']
                )
                print(f"Connected to database: {self.config['database']}")
            except Exception as e:
                self.connection = None
                raise Exception(
                    f"Connection error! Check credentials in .env\n"
                    f"Make sure the container is running: docker compose up -d"
                ) from e
        return self.connection
    
    def is_connected(self):
        return self.connection is not None
    
    def close(self):
        if self.connection:
            self.connection.close()
            self.connection = None
            print("Connection closed.")

## Executor class

In [18]:
class PostgresExecutor:
    """Class responsible for SQL operations."""

    def __init__(self, connection):
        self.connection = connection

    def execute(self, query, params=None):
        cursor = self.connection.cursor()
        cursor.execute(query, params)
        self.connection.commit()
        cursor.close()

    def fetch(self, query, params=None):
        cursor = self.connection.cursor()
        cursor.execute(query, params)
        results = cursor.fetchall()
        cursor.close()
        return results

    def count(self, table):
        result = self.fetch(f'SELECT COUNT(*) FROM {table}')
        return result[0][0]

## Connection to PostgreSQL


In [19]:
db = PostgresDatabase(DB_CONFIG)

try:
    connection = db.connect()
    executor = PostgresExecutor(connection)
    print("Ready to ingest!")
except Exception as e:
    print(f"Connection failed: {e}")

Connected to database: labdb
Ready to ingest!


## Create table

In [20]:
create_table_query = """
CREATE TABLE IF NOT EXISTS diabetes (
    id SERIAL PRIMARY KEY,
    pregnancies INTEGER,
    glucose INTEGER,
    blood_pressure INTEGER,
    skin_thickness INTEGER,
    insulin INTEGER,
    body_mass_index FLOAT,
    diabetes_pedigree_function FLOAT,
    age INTEGER,
    outcome TEXT,
    clinic_region TEXT,
    care_path TEXT,
    patient_segment TEXT,
    source_file TEXT
);
"""

executor.execute(create_table_query)
print("Table 'diabetes' created successfully!")

Table 'diabetes' created successfully!


## Read the cleaned CSV

In [21]:
df = pd.read_csv(CLEANED_PATH)


In [22]:
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 771 rows, 13 columns


In [23]:
df.head(3)

,pregnancies,glucose,blood pressure,skin thickness,insulin,body mass index,diabetes pedigree function,age,outcome,clinic region,care path,patient segment,source_file
0,6,148.0,72.0,35.0,NaN,NaN,0.627,50.0,Positive,North,High-Risk,Adult,diabetes-dirty.csv
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31.0,Negative,South,Routine Follow-up,Senior,diabetes-dirty.csv
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32.0,Positive,North,Urgent Monitoring,Mid-Age,diabetes-dirty.csv


## Map a raw row into a PostgreSQL row

In [24]:
def map_record(row):
    def to_int(val):
        if pd.isna(val):
            return None
        return int(float(val))
    
    def to_float(val):
        if pd.isna(val):
            return None
        return float(val)
    
    def to_str(val):
        if pd.isna(val):
            return None
        return str(val)

    return (
        to_int(row['pregnancies']),
        to_int(row['glucose']),
        to_int(row['blood pressure']),
        to_int(row['skin thickness']),
        to_int(row['insulin']),
        to_float(row['body mass index']),
        to_float(row['diabetes pedigree function']),
        to_int(row['age']),
        to_str(row['outcome']),
        to_str(row['clinic region']),
        to_str(row['care path']),
        to_str(row['patient segment']),
        to_str(row['source_file'])
    )

## Ingest data

In [25]:
def ingest_data(executor, df, limit=None):
    """Insert rows from the DataFrame into the diabetes table."""
    if limit is not None:
        rows = df.head(limit).copy()
    else:
        rows = df.copy()

    insert_query = """
        INSERT INTO diabetes (
            pregnancies, glucose, blood_pressure, skin_thickness,
            insulin, body_mass_index, diabetes_pedigree_function, age,
            outcome, clinic_region, care_path, patient_segment, source_file
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    count = 0
    for _, row in rows.iterrows():
        postgres_row = map_record(row)
        executor.execute(insert_query, postgres_row)
        count += 1

    return count

In [26]:
print(df.head(1).to_dict('records'))

[{'pregnancies': 6, 'glucose': 148.0, 'blood pressure': 72.0, 'skin thickness': 35.0, 'insulin': nan, 'body mass index': nan, 'diabetes pedigree function': 0.627, 'age': 50.0, 'outcome': 'Positive', 'clinic region': 'North', 'care path': 'High-Risk', 'patient segment': 'Adult', 'source_file': 'diabetes-dirty.csv'}]


## Small validation checks


In [27]:
# Clear table and reset id
executor.execute('TRUNCATE TABLE diabetes RESTART IDENTITY')
# Test with 5 rows first
n = ingest_data(executor, df, limit=5)
print(f"Test ingestion: {n} rows inserted")
print(f"Total in table: {executor.count('diabetes')}")

Test ingestion: 5 rows inserted
Total in table: 5


## Full ingestion

In [28]:
# Clear test rows
executor.execute('TRUNCATE TABLE diabetes RESTART IDENTITY')

# Ingest full dataset
n_total = ingest_data(executor, df)
print(f"Full ingestion complete: {n_total} rows inserted")

Full ingestion complete: 771 rows inserted


In [29]:
#  Validation queries:

# Total rows
total = executor.count('diabetes')
print(f"Total rows in table: {total}")

# Count by outcome
by_outcome = executor.fetch('SELECT outcome, COUNT(*) FROM diabetes GROUP BY outcome')
print("\nCount by outcome:")
for row in by_outcome:
    print(f"  {row[0]}: {row[1]}")

# Count by clinic region
by_region = executor.fetch('SELECT clinic_region, COUNT(*) FROM diabetes GROUP BY clinic_region')
print("\nCount by clinic region:")
for row in by_region:
    print(f"  {row[0]}: {row[1]}")

# Sample row
sample = executor.fetch('SELECT * FROM diabetes LIMIT 1')
print(f"\nSample row: {sample[0]}")

Total rows in table: 771

Count by outcome:
  None: 43
  Positive: 286
  Negative: 442

Count by clinic region:
  None: 47
  South: 363
  North: 361

Sample row: (1, 6, 148, 72, 35, None, None, 1, 50, 'Positive', 'North', 'High-Risk', 'Adult', 'diabetes-dirty.csv')


In [30]:
sample = executor.fetch('SELECT diabetes_pedigree_function FROM diabetes LIMIT 5')
print(sample)

[(1,), (0,), (1,), (0,), (2,)]


In [ ]:
db.close()

Connection closed.
